# 02 Window Dataset And Feature Table
Этапы 6-7: EDA по target и построение dataset history_1..history_3 -> target с сохранением артефактов.

In [1]:
import pandas as pd

from _shared_notebook_utils import DATA_DIR, ROOT, ensure_dirs, save_json, save_pickle

ensure_dirs()
print('ROOT =', ROOT)
print('DATA_DIR =', DATA_DIR)

ROOT = C:\Users\Dmitry\code-projects\diploma-crop-rotation
DATA_DIR = C:\Users\Dmitry\code-projects\diploma-crop-rotation\artifacts\data


## 1) Загрузка подготовленного датасета

In [2]:
input_path = DATA_DIR / '01_df_prepared_filtered.pkl'
if not input_path.exists():
    fallback_path = ROOT / 'artifacts' / 'research_checkpoint' / 'baseline' / 'window_df_filtered.pkl'
    if not fallback_path.exists():
        raise FileNotFoundError('No input found. Run notebook 01 first or ensure fallback checkpoint exists.')
    df_input = pd.read_pickle(fallback_path)
    input_source = 'fallback_checkpoint_window_df_filtered'
else:
    df_input = pd.read_pickle(input_path)
    input_source = '01_df_prepared_filtered.pkl'

print('input_source =', input_source)
print('df_input shape =', df_input.shape)

input_source = 01_df_prepared_filtered.pkl
df_input shape = (38511210, 11)


## 2) Построение window dataset (если не готов)

In [3]:
required_window_cols = {'history_1', 'history_2', 'history_3', 'target'}
if required_window_cols.issubset(set(df_input.columns)):
    window_df = df_input.copy()
    window_source = 'already_window_like'
else:
    group_cols = sorted([c for c in df_input.columns if c.startswith('CDL') and c.endswith('_group') and c[3:7].isdigit()])
    if len(group_cols) < 4:
        raise ValueError('Need at least 4 yearly *_group columns to build history window of size 3.')

    fixed_cols = [c for c in ['CSBID', 'CSBACRES', 'STATEFIPS', 'CNTYFIPS', 'INSIDE_X', 'INSIDE_Y'] if c in df_input.columns]
    parts = []
    for i in range(len(group_cols) - 3):
        h1, h2, h3, tgt = group_cols[i:i+4]
        part = df_input[fixed_cols + [h1, h2, h3, tgt]].rename(columns={
            h1: 'history_1',
            h2: 'history_2',
            h3: 'history_3',
            tgt: 'target'
        })
        part['target_year'] = int(tgt[3:7])
        parts.append(part)

    window_df = pd.concat(parts, axis=0, ignore_index=True)
    window_source = 'built_from_CDL_group_columns'

print('window_source =', window_source)
print('window_df shape =', window_df.shape)
display(window_df.head(3))

window_source = already_window_like
window_df shape = (38511210, 11)


,CSBID,target_year,history_1,history_2,history_3,target,CSBACRES,STATEFIPS,CNTYFIPS,INSIDE_X,INSIDE_Y
0,351724000000036,2020,wheat,wheat,sorghum,sorghum,20.942217,35,021,-664007.6193,1.445271e+06
1,351724000000037,2020,wheat,wheat,sorghum,sorghum,10.916051,35,021,-664304.1197,1.445202e+06
2,351724000000044,2020,forage_hay,forage_hay,other_cereals,fallow,17.243968,35,021,-664688.1027,1.445128e+06


## 3) Target EDA и фильтрация редких классов

In [4]:
RARE_SHARE_THRESHOLD = 0.01

target_summary = (
    window_df['target']
    .astype('string')
    .value_counts(dropna=False)
    .rename_axis('target')
    .reset_index(name='count')
)
target_summary['share'] = target_summary['count'] / target_summary['count'].sum()
rare_target_classes = target_summary.loc[target_summary['share'] < RARE_SHARE_THRESHOLD, 'target'].astype(str).tolist()

window_df_filtered = window_df.loc[~window_df['target'].astype('string').isin(rare_target_classes)].copy()

print('unique target classes before:', target_summary.shape[0])
print('rare target classes:', rare_target_classes)
print('window_df_filtered shape:', window_df_filtered.shape)
display(target_summary.head(15))

unique target classes before: 9
rare target classes: []
window_df_filtered shape: (38511210, 11)


,target,count,share
0,corn,11752817,0.305179
1,soybeans,10970284,0.284859
2,wheat,5648151,0.146663
3,forage_hay,3740641,0.097131
4,cotton,1849867,0.048035
5,fallow,1802576,0.046807
6,sorghum,1158288,0.030077
7,other_cereals,1008139,0.026178
8,legumes,580447,0.015072


## 4) Сохранение артефактов этапа

In [5]:
window_path = DATA_DIR / '02_window_df_filtered.pkl'
summary_path = DATA_DIR / '02_target_summary.pkl'
meta_path = DATA_DIR / '02_meta.json'

window_size_mb = save_pickle(window_df_filtered, window_path)
save_pickle(target_summary, summary_path)

meta = {
    'input_source': input_source,
    'window_source': window_source,
    'rare_share_threshold': RARE_SHARE_THRESHOLD,
    'rare_target_classes': rare_target_classes,
    'window_rows': int(window_df.shape[0]),
    'window_filtered_rows': int(window_df_filtered.shape[0]),
    'window_filtered_cols': int(window_df_filtered.shape[1])
}
save_json(meta, meta_path)

print('Saved:', window_path, f'({window_size_mb:.2f} MB)')
print('Saved:', summary_path)
print('Saved:', meta_path)

Saved: C:\Users\Dmitry\code-projects\diploma-crop-rotation\artifacts\data\02_window_df_filtered.pkl (4263.56 MB)
Saved: C:\Users\Dmitry\code-projects\diploma-crop-rotation\artifacts\data\02_target_summary.pkl
Saved: C:\Users\Dmitry\code-projects\diploma-crop-rotation\artifacts\data\02_meta.json
